# Bài tập 6 — Lớp hiếm và xác suất tiên nghiệm
Giữ toàn bộ 180 tin thường và chỉ lấy 20 tin rác đầu tiên theo thứ tự CSV. Chia tập có stratify, in accuracy và recall lớp rác; thêm mốc luôn đoán thường để giải thích bẫy accuracy.

Chọn **Run All** để chạy độc lập notebook này. Trên Colab, tải CSV cần dùng vào Files hoặc thư mục `data/`.


In [1]:
from pathlib import Path

# Chạy từ thư mục bài, code/, hoặc thư mục chứa CSV đã tải lên Colab.
thu_muc_hien_tai = Path.cwd().resolve()
thu_muc_du_an = thu_muc_hien_tai
for thu_muc in (thu_muc_hien_tai, *thu_muc_hien_tai.parents):
    if (thu_muc / "data" / "tin_nhan.csv").is_file():
        thu_muc_du_an = thu_muc
        break

def tep_du_lieu(ten):
    for duong_dan in (thu_muc_du_an / "data" / ten, thu_muc_hien_tai / ten):
        if duong_dan.is_file():
            return duong_dan
    raise FileNotFoundError(f"Không tìm thấy {ten}. Hãy giữ thư mục data/ hoặc tải CSV lên Colab.")

thu_muc_hinh = thu_muc_du_an / "figures"
thu_muc_hinh.mkdir(exist_ok=True)


## Mã và kết quả

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix

df = pd.read_csv(tep_du_lieu("tin_nhan.csv"))
tin_thuong = df[df["la_rac"] == 0]
tin_rac = df[df["la_rac"] == 1].head(20)
# Ghép nhóm thường trước, nhóm rác sau; đặt lại chỉ mục để tái lập kết quả.
df_lech = pd.concat([tin_thuong, tin_rac], ignore_index=True)
print("Số tin theo lớp trong bộ mới:")
print(df_lech["la_rac"].value_counts().sort_index())

vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(df_lech["noi_dung"])
y = df_lech["la_rac"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
model = MultinomialNB(alpha=1.0).fit(X_train, y_train)
pred = model.predict(X_test)
accuracy = accuracy_score(y_test, pred)
recall_rac = recall_score(y_test, pred, pos_label=1, zero_division=0)
tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()

print(f"Tập học: {len(y_train)}; tập kiểm tra: {len(y_test)}")
print(f"Tiên nghiệm lớp thường: {(y_train == 0).mean():.4f}; lớp rác: {(y_train == 1).mean():.4f}")
print(f"TN = {tn}; FP = {fp}; FN = {fn}; TP = {tp}")
print(f"Accuracy = {accuracy:.4f}")
print(f"Recall lớp rác = {recall_rac:.4f}")
print(f"Luôn đoán thường: accuracy = {(y_test == 0).mean():.4f}; recall lớp rác = 0.0000")
print(f"Nhận xét: Bộ lọc bắt được {tp}/{tp + fn} tin rác trong tập kiểm tra. Cần đọc recall lớp rác cùng accuracy để biết khả năng lọc thực tế.")
print("Vì 90% dữ liệu là tin thường, chỉ đoán thường đã đạt accuracy 90% nhưng bỏ lọt toàn bộ tin rác. Accuracy cao riêng lẻ không chứng minh bộ lọc tốt khi các lớp mất cân bằng.")
print("Tiên nghiệm nghiêng về lớp thường, còn các xác suất từ của lớp rác chỉ được học từ ít tin; do đó kết quả cũng nhạy với các tin rác được chọn.")

Số tin theo lớp trong bộ mới:
la_rac
0    180
1     20
Name: count, dtype: int64
Tập học: 150; tập kiểm tra: 50
Tiên nghiệm lớp thường: 0.9000; lớp rác: 0.1000
TN = 45; FP = 0; FN = 1; TP = 4
Accuracy = 0.9800
Recall lớp rác = 0.8000
Luôn đoán thường: accuracy = 0.9000; recall lớp rác = 0.0000
Nhận xét: Bộ lọc bắt được 4/5 tin rác trong tập kiểm tra. Cần đọc recall lớp rác cùng accuracy để biết khả năng lọc thực tế.
Vì 90% dữ liệu là tin thường, chỉ đoán thường đã đạt accuracy 90% nhưng bỏ lọt toàn bộ tin rác. Accuracy cao riêng lẻ không chứng minh bộ lọc tốt khi các lớp mất cân bằng.
Tiên nghiệm nghiêng về lớp thường, còn các xác suất từ của lớp rác chỉ được học từ ít tin; do đó kết quả cũng nhạy với các tin rác được chọn.
